### Installation

[Traning](https://oneclickamd.ai/github.com/azheraly/Aqal-First-Urdu-Reasoning-Large-Language-Model/blob/main/nb/qwen-sft.ipynb)


In [ ]:
%%bash
python -m pip install -qU uv --root-user-action=ignore

ROCM_TAG="$({ command -v amd-smi >/dev/null 2>&1 && amd-smi version 2>/dev/null | awk -F'ROCm version: ' 'NF>1{split($2,a,"."); print "rocm"a[1]"."a[2]; ok=1; exit} END{exit !ok}'; } || { [ -r /opt/rocm/.info/version ] && awk -F. '{print "rocm"$1"."$2; exit}' /opt/rocm/.info/version; } || { command -v hipconfig >/dev/null 2>&1 && hipconfig --version 2>/dev/null | awk -F': *' '/HIP version/{split($2,a,"."); print "rocm"a[1]"."a[2]; ok=1; exit} END{exit !ok}'; } || { command -v dpkg-query >/dev/null 2>&1 && ver="$(dpkg-query -W -f='${Version}\n' rocm-core 2>/dev/null)" && [ -n "$ver" ] && awk -F'[.-]' '{print "rocm"$1"."$2; exit}' <<<"$ver"; } || { command -v rpm >/dev/null 2>&1 && ver="$(rpm -q --qf '%{VERSION}\n' rocm-core 2>/dev/null)" && [ -n "$ver" ] && awk -F'[.-]' '{print "rocm"$1"."$2; exit}' <<<"$ver"; })"
[ -n "$ROCM_TAG" ] || { echo "Could not detect ROCm. Install ROCm first or set ROCM_TAG manually."; exit 1; }
case "$ROCM_TAG" in
  rocm6.[0-4]|rocm7.[02]) T="$ROCM_TAG" ;;
  rocm6.*) T="rocm6.4" ;;
  *) T="rocm7.1" ;;
esac
pip install bitsandbytes
PYTORCH_INDEX_URL="https://download.pytorch.org/whl/${T}"
uv pip install --system -U --force-reinstall \
    torch torchvision torchaudio triton-rocm \
    --index-url "$PYTORCH_INDEX_URL"
uv pip install --system cut-cross-entropy torchao --no-deps
uv pip install --system -U --no-deps "unsloth[amd]" "unsloth_zoo[amd]"
uv pip install --system --no-deps -r "$(python -c 'import pathlib,site;print(next(p for r in [*site.getsitepackages(),site.getusersitepackages()] if (p:=pathlib.Path(r,"studio/backend/requirements/no-torch-runtime.txt")).exists()))')" torchao
uv pip install --system --no-deps -U "tokenizers>=0.22.0,<=0.23.0"


In [ ]:
!uv pip install --system -qqq sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer "transformers==4.56.2"
!uv pip install --system -qqq --no-deps accelerate peft "trl==0.22.2"


In [ ]:
from huggingface_hub import notebook_login, list_repo_files, snapshot_download

notebook_login()

### Unsloth


In [2]:
%env UNSLOTH_RETURN_LOGITS = 1 # Run this to disable CCE since it is not supported for CPT

env: UNSLOTH_RETURN_LOGITS=1 # Run this to disable CCE since it is not supported for CPT


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
dtype = (
    None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
)
load_in_4bit = False  # Use 4bit quantization to reduce memory usage. Can be False.
load_in_8bit = False  # Use 8bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    # Can select any from the below:
    # "unsloth/Qwen2.5-0.5B", "unsloth/Qwen2.5-1.5B", "unsloth/Qwen2.5-3B"
    # "unsloth/Qwen2.5-14B",  "unsloth/Qwen2.5-32B",  "unsloth/Qwen2.5-72B",
    # And also all Instruct versions and Math. Coding versions!
    model_name="azherali/Aqal-1.0-8B-Lora",  # Choose ANY
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    load_in_8bit=load_in_8bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

<a name="Data"></a>

### Data Prep

We now use the Qwen-3 format for conversation style finetunes.Qwen-3 renders multi turn conversations like below:

"""<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>
"""

We use our get_chat_template function to get the correct chat template. We support zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3 and more.


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3-instruct",
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("azherali/UrduMath", split="train")

In [ ]:
def generate_conversation(examples):
    problems = examples["problem"]
    solutions = examples["solution"]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append(
            [
                {"role": "user", "content": problem},
                {"role": "assistant", "content": solution},
            ]
        )
    return {
        "conversations": conversations,
    }


dataset = dataset.map(generate_conversation, batched=True)

We now have to apply the chat template for Qwen-3 onto the conversations, and save it to text.


In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {
        "text": texts,
    }


dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
dataset[100]["text"]

In [ ]:
from transformers import TrainingArguments
from unsloth import UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=4,
    args=UnslothTrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=8,
        # Use warmup_ratio and num_train_epochs for longer runs!
        # max_steps=80,
        # warmup_steps=10,
        warmup_ratio=0.1,
        num_train_epochs=2,
        # Select a 2 to 10x smaller learning rate for the embedding matrices!
        learning_rate=5e-5,
        embedding_learning_rate=1e-5,
        logging_steps=20,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="Riazi-8B-Lora",  # Change this to your preferred output directory
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        report_to="none",  # Use wandb etc
        push_to_hub=True,
        hub_strategy="checkpoint",
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
print(f"Start training")

try:
    repo_id = "azherali/Riazi-8B-Lora"  # Change to your repo ID
    files = list_repo_files(repo_id)
    checkpoints = sorted(
        [f for f in files if f.startswith("last-checkpoint")],
    )
    if len(checkpoints) == 0:
        print("No checkpoints — starting new training.")
        trainer_stats = trainer.train()
    else:
        local_dir = snapshot_download(repo_id, allow_patterns="last-checkpoint/*")
        print("Resuming from:", local_dir)
        trainer_stats = trainer.train(
            resume_from_checkpoint=f"{local_dir}/last-checkpoint"
        )
except Exception as e:
    print("Repo not found or empty — starting fresh training.", e)
    trainer_stats = trainer.train()

<a name="Save"></a>

### Saving, loading finetuned models

To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!


In [ ]:
# Just LoRA adapters
if False:
    model.save_pretrained("Riazi-8B-Lora")  # Local saving
    tokenizer.save_pretrained("Riazi-8B-Lora")
if True:
    model.push_to_hub("azherali/Riazi-8B-Lora")
    tokenizer.push_to_hub("azherali/Riazi-8B-Lora")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback.


In [ ]:
# # Merge to 16bit
# if False:
#     model.save_pretrained_merged(
#         "azherali/Riazi-8B",
#         tokenizer,
#         save_method="merged_16bit",
#     )
# if True:
#     model.push_to_hub_merged(
#         "azherali/Riazi-8B",
#         tokenizer,
#         save_method="merged_16bit",
#     )

# # Merge to 4bit
# if False:
#     model.save_pretrained_merged(
#         "azherali/Riazi-8B",
#         tokenizer,
#         save_method="merged_4bit",
#     )
# if False:
#     model.push_to_hub_merged(
#         "azherali/Riazi-8B",
#         tokenizer,
#         save_method="merged_4bit",
#     )

# Merge to 16bit
if False:
    model.save_pretrained_merged(
        "azherali/Riazi-8B",
        tokenizer,
        save_method="merged_16bit",
    )
if True:
    model.push_to_hub_merged(
        "azherali/Riazi-8B",
        tokenizer,
        save_method="merged_16bit",
    )

# Merge to 4bit
if False:
    model.save_pretrained_merged(
        "azherali/Riazi-8B",
        tokenizer,
        save_method="merged_4bit",
    )
if False:
    model.push_to_hub_merged(
        "azherali/Riazi-8B",
        tokenizer,
        save_method="merged_4bit",
    )

### GGUF / llama.cpp Conversion

To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):

- `q8_0` - Fast conversion. High resource use, but generally acceptable.
- `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
- `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.


In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf(
        "mistral_v0_finetune",
        tokenizer,
    )
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/mistral_v0_finetune", tokenizer, token="YOUR_HF_TOKEN"
    )

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf(
        "mistral_v0_finetune", tokenizer, quantization_method="f16"
    )
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/mistral_v0_finetune",
        tokenizer,
        quantization_method="f16",
        token="YOUR_HF_TOKEN",
    )

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf(
        "mistral_v0_finetune", tokenizer, quantization_method="q4_k_m"
    )
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/mistral_v0_finetune",
        tokenizer,
        quantization_method="q4_k_m",
        token="YOUR_HF_TOKEN",
    )
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/mistral_v0_finetune",
        tokenizer,
        quantization_method="q5_k_m",
        token="YOUR_HF_TOKEN",
    )

### Inference


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
dtype = (
    None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
)
load_in_4bit = False  # Use 4bit quantization to reduce memory usage. Can be False.
load_in_8bit = False  # Use 8bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="azherali/Riazi-8B",  # Choose ANY
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    load_in_8bit=load_in_8bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference
messages = [
    {
        "role": "user",
        "content": "پانچ بچوں نے 20 چاکلیٹس برابر بانٹیں۔ ہر بچے کو کتنی چاکلیٹس ملیں گی؟",
    }
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # Must add for generation
)

from transformers import TextStreamer

_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    temperature=0.6,
    top_p=0.95,
    top_k=20,  # For non thinking
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

In [47]:
import pandas as pd

df = pd.read_csv("../results/qalb_evaluation_scores.csv")
len(df)

250

In [48]:
df

,correctness,reasoning,UrduLanguageFluency,clarity,completeness,row_idx
0,1,1,4,3,2,0
1,3,2,4,3,3,1
2,1,1,3,3,3,2
3,5,5,4,5,5,3
4,1,1,3,2,1,4
...,...,...,...,...,...,...
245,1,1,2,1,1,245
246,1,1,2,1,1,246
247,5,5,5,5,5,247
248,1,1,1,1,1,248


In [ ]:
import pandas as pd

df = pd.read_csv("../results/alif_evaluation_scores.csv")

metrics = ["correctness", "reasoning", "UrduLanguageFluency", "clarity", "completeness"]

for metric in metrics:
    mean = df[metric].mean()
    std = df[metric].std()
    print(f"{metric}: {mean:.2f} ± {std:.2f}")

correctness: 2.27 ± 1.84
reasoning: 2.23 ± 1.77
UrduLanguageFluency: 3.52 ± 1.02
clarity: 2.83 ± 1.52
completeness: 2.67 ± 1.62


In [ ]:
import pandas as pd

df = pd.read_csv("../results/qalb_evaluation_scores.csv")

metrics = ["correctness", "reasoning", "UrduLanguageFluency", "clarity", "completeness"]

for metric in metrics:
    mean = df[metric].mean()
    std = df[metric].std()
    print(f"{metric}: {mean:.2f} ± {std:.2f}")

correctness: 2.27 ± 1.84
reasoning: 2.29 ± 1.78
UrduLanguageFluency: 3.15 ± 1.01
clarity: 2.72 ± 1.61
completeness: 2.67 ± 1.70


In [ ]:
import pandas as pd

df = pd.read_csv("../results/aqal_evaluation_scores.csv")

metrics = ["correctness", "reasoning", "UrduLanguageFluency", "clarity", "completeness"]

for metric in metrics:
    mean = df[metric].mean()
    std = df[metric].std()
    print(f"{metric}: {mean:.2f} ± {std:.2f}")

correctness: 3.87 ± 1.77
reasoning: 3.90 ± 1.69
UrduLanguageFluency: 4.14 ± 0.80
clarity: 4.20 ± 1.29
completeness: 4.19 ± 1.34


In [ ]:
import pandas as pd

files = {
    "Alif-1.0-8B-Instruct": "alif_evaluation_scores.csv",
    "Qalb-1.0-8B-Instruct": "qalb_evaluation_scores.csv",
    "Riazi-8B": "aqal_evaluation_scores.csv",
}

metrics = ["correctness", "reasoning", "UrduLanguageFluency", "clarity", "completeness"]

rows = []

for model, file in files.items():
    df = pd.read_csv(f"../results/{file}")

    row = {"Models": model}

    for metric in metrics:
        mean = df[metric].mean()
        std = df[metric].std()

        row[metric] = f"{mean:.2f} ± {std:.2f}"

    rows.append(row)

results = pd.DataFrame(rows)

print(results)

                 Models  correctness    reasoning UrduLanguageFluency  \
0  Alif-1.0-8B-Instruct  2.27 ± 1.84  2.23 ± 1.77         3.52 ± 1.02   
1  Qalb-1.0-8B-Instruct  2.27 ± 1.84  2.29 ± 1.78         3.15 ± 1.01   
2  Aqal-1.0-8B-Instruct  3.87 ± 1.77  3.90 ± 1.69         4.14 ± 0.80   

       clarity completeness  
0  2.83 ± 1.52  2.67 ± 1.62  
1  2.72 ± 1.61  2.67 ± 1.70  
2  4.20 ± 1.29  4.19 ± 1.34  


In [ ]:
import pandas as pd

files = {
    "Alif-1.0-8B-Instruct": "alif_evaluation_scores.csv",
    "Qalb-1.0-8B-Instruct": "qalb_evaluation_scores.csv",
    "Riazi-8B": "aqal_evaluation_scores.csv",
}

metrics = ["correctness", "reasoning", "UrduLanguageFluency", "clarity", "completeness"]

rows = []

for model, file in files.items():
    df = pd.read_csv(f"../results/{file}")

    row = {"Model": model}

    metric_means = []

    for metric in metrics:
        mean = df[metric].mean()
        std = df[metric].std()

        metric_means.append(mean)

        row[metric] = f"{mean:.2f} ± {std:.2f}"

    # Overall score = average of metric means
    overall = sum(metric_means) / len(metric_means)
    row["Mean Score"] = f"{overall:.2f}"

    rows.append(row)

results = pd.DataFrame(rows)

print(results)

                  Model  correctness    reasoning UrduLanguageFluency  \
0  Alif-1.0-8B-Instruct  2.27 ± 1.84  2.23 ± 1.77         3.52 ± 1.02   
1  Qalb-1.0-8B-Instruct  2.27 ± 1.84  2.29 ± 1.78         3.15 ± 1.01   
2  Aqal-1.0-8B-Instruct  3.87 ± 1.77  3.90 ± 1.69         4.14 ± 0.80   

       clarity completeness Mean Score  
0  2.83 ± 1.52  2.67 ± 1.62       2.70  
1  2.72 ± 1.61  2.67 ± 1.70       2.62  
2  4.20 ± 1.29  4.19 ± 1.34       4.06  


In [ ]:
import pandas as pd

df = pd.read_csv("../results/aqal_evaluation_scores.csv")
df.head()

,correctness,reasoning,UrduLanguageFluency,clarity,completeness
0,5,5,4,5,5
1,5,5,4,5,5
2,5,5,5,5,5
3,1,1,2,2,1
4,1,1,3,1,1


In [ ]:
scs = (df["completeness"].mean() / 5) * 100
print(f"SCS = {scs:.1f}%")

SCS = 83.8%


In [ ]:
uop = (df["UrduLanguageFluency"].mean() / 5) * 100
print(f"UOP = {uop:.1f}%")

UOP = 82.7%
